In [ ]:
from fedn.utils.helpers.helpers import get_helper
from ultralytics import YOLO
import torch
import collections
import numpy as np
import hopsworks
import os
from PIL import Image
import train

HELPER_MODULE = "numpyhelper"
helper = get_helper(HELPER_MODULE)


In [ ]:
model = train.load_parameters("weights/face_finder_best.npz")
params = {
    'data': '/hopsfs/Jupyter/yolov8-face/data/widerface.yaml',
    'epochs': 1,
    'batch': 32,
    'imgsz': 640,
    'device': 0,
    'resume': False,
    'workers': 0,
    'cache': "ram",
    'amp': True,
}

print(params)

In [ ]:
model.train(**params)

### Model Evaluation
Predict bounding boxes for an example image and save the output image with the model.

In [ ]:
img_path = "data/images/bus.jpg"
results = model.predict(
    img_path,
    imgsz=640,
    conf=0.75,
    iou=0.7,
    device=0,
    verbose=False
)

img = results[0].plot()  # BGR numpy array
img = Image.fromarray(img[..., ::-1])  # Convert to RGB for PIL

base, _ = os.path.splitext(os.path.basename(img_path))
output_filename = f"./{model_dir}/images/{base}-faces-detected.png"
output_path = os.path.abspath(output_filename)
img.save(output_path, format="PNG")

### Save Trained Model to Hopsworks Model Registry

Save the serialized model, evaluation images, and any metrics to the model registry

In [ ]:
mr = hopsworks.login().get_model_registry()
model_dir = "mr_model"
os.makedirs(f"{model_dir}/images", exist_ok=True)
save_parameters(model, f"./{model_dir}/fine-tuned-model.npz")

metrics = {
    "epochs": params['epochs'],
    "batch": params['batch'],    
}

faces_model = mr.python.create_model(
    name="facerecognition", 
    metrics=metrics,
    description="Yolo-v8 face recognition model", 
)

# Save the model to the specified directory
faces_model.save(model_dir)